# 創建資料表

In [ ]:
import MySQLdb

# host 必須寫 127.0.0.1，不能寫 localhost。
# MySQLdb 看到 "localhost" 會改走 Unix socket（/tmp/mysql.sock），
# 但 MySQL 跑在 Docker 容器裡，本機沒有那個 socket 檔，會直接連線失敗。
# 寫 127.0.0.1 才會走 TCP，經由 compose.yaml 的 3306 埠對應進容器。
# 四個參數依序是：host, user, password, database（密碼見 mysql-learning/.env）
db = MySQLdb.connect("127.0.0.1", "root", "root", "mydatabase")

cursor = db.cursor()

## 1.如果 STUDENTS 資料表存在，將 STUDENTS 資料表丟棄
# 原本寫的是 DROP DATABASE IF EXISTS STUDENTS。
# DATABASE 丟的是「資料庫」，但 STUDENTS 是「資料表」，
# 所以那行永遠不會生效（也不會報錯，因為有 IF EXISTS），
# 導致第二次執行本格時撞上 "Table 'STUDENTS' already exists"。
cursor.execute("DROP TABLE IF EXISTS STUDENTS")


## 2.創建STUDENTS資料表，資料表欄位如下
'''
| ID INT | NAME CHAR(20) | GENDER CHAR(20) | CHINESE CHAR(20) | ENGLISH CHAR(20) | MATH CHAR(20) | SOCIAL_SCIENCE CHAR(20) | SCIENCE CHAR(20) |

PRIMARY KEY = ID
CHARSET = utf8mb4
'''

# 兩處對 MySQL 8.4 的調整：
#   1. INT(11) 的括號叫「顯示寬度」，MySQL 8 已棄用，會跳 deprecation 警告，改成 INT。
#   2. CHARSET=utf8 在 MySQL 8 是 utf8mb3 的別名，只能存 3 bytes，存不了 emoji
#      與部分罕用漢字，也已棄用；改用 utf8mb4（真正的完整 UTF-8）。
sql = """CREATE TABLE STUDENTS (
         ID INT NOT NULL AUTO_INCREMENT,
         NAME  CHAR(20),
         GENDER  CHAR(20),
         CHINESE CHAR(20),
         ENGLISH CHAR(20),
         MATH CHAR(20),
         SOCIAL_SCIENCE CHAR(20),
         SCIENCE  CHAR(20),

         PRIMARY KEY  (ID)
         ) DEFAULT CHARSET=utf8mb4"""
cursor.execute(sql)

cursor.close()
db.close()
print("STUDENTS 資料表建立完成")

# 手key學生成績到資料庫內  
請手動輸入以下兩位學生成績  
name: isaac, gender: m, chinese: 60, english: 72, math: 32, social_science: 52, science: 86  
name: amy, gender: f, chinese: 50, english: 22, math: 80, social_science: 15, science: 93

In [ ]:
import MySQLdb

# host 必須寫 127.0.0.1，不能寫 localhost。
# MySQLdb 看到 "localhost" 會改走 Unix socket（/tmp/mysql.sock），
# 但 MySQL 跑在 Docker 容器裡，本機沒有那個 socket 檔，會直接連線失敗。
# 寫 127.0.0.1 才會走 TCP，經由 compose.yaml 的 3306 埠對應進容器。
# 四個參數依序是：host, user, password, database（密碼見 mysql-learning/.env）
db = MySQLdb.connect("127.0.0.1", "root", "root", "mydatabase")

cursor = db.cursor()

print("enter students score")
while True:

    student_name = input("enter name: ")
    student_gender = input("enter gender: ")
    student_chinese = input("enter chinese score: ")
    student_english = input("enter english score: ")
    student_math = input("enter math score: ")
    student_social_science = input("enter social science score: ")
    student_science = input("enter science score: ")

    ## 3.將student_name, student_gender, student_chinese, ......插入到資料庫
    x = (student_name, student_gender, student_chinese, student_english, student_math, student_social_science, student_science)
    sql = """INSERT INTO STUDENTS(
         name, gender, chinese, english, math, social_science, science)
         VALUES ( %s, %s, %s, %s, %s, %s, %s)"""

    try:
        cursor.execute(sql, x)
        db.commit()
        print("  已寫入 {}".format(student_name))
    # 原本是裸 except（except: 後面不接任何型別），它會吞掉「所有」例外，
    # 包含 Ctrl-C 與打錯字造成的 NameError，畫面上什麼都不會顯示，
    # 只會靜默 rollback，讓人誤以為寫入成功。改成只接資料庫錯誤並印出原因。
    except MySQLdb.Error as e:
        db.rollback()
        print("  寫入失敗：{}".format(e))

    again = input("continue(y/n)? ")
    # 原本寫 again[0]，若直接按 Enter（空字串）會 IndexError。
    # 用切片 again[:1] 取第一個字元，空字串時得到 ""，不會出錯。
    if again[:1].lower() == "n":
        break

cursor.close()
db.close()

# 查詢目前資料庫所有內容

In [ ]:
import MySQLdb

# host 必須寫 127.0.0.1，不能寫 localhost。
# MySQLdb 看到 "localhost" 會改走 Unix socket（/tmp/mysql.sock），
# 但 MySQL 跑在 Docker 容器裡，本機沒有那個 socket 檔，會直接連線失敗。
# 寫 127.0.0.1 才會走 TCP，經由 compose.yaml 的 3306 埠對應進容器。
# 四個參數依序是：host, user, password, database（密碼見 mysql-learning/.env）
db = MySQLdb.connect("127.0.0.1", "root", "root", "mydatabase")

cursor = db.cursor()

sql = "SELECT * FROM STUDENTS"

## 4.查詢目前資料庫所有內容
try:
    cursor.execute(sql)
    results = cursor.fetchall()

    for row in results:
        student_id = row[0]      # <------ Don't forget this !!!
        student_name = row[1]
        student_gender = row[2]
        student_chinese = row[3]
        student_english = row[4]
        student_math = row[5]
        student_social_science = row[6]
        student_science = row[7]
        print("name: {}, gender: {}, chinese: {}, english: {}, math: {}, social_science: {}, science: {}"
              .format(student_name, student_gender, student_chinese, student_english, student_math, student_social_science, student_science))

    print("共 {} 筆".format(len(results)))
# 原本是 except: print("Error: unable to fecth data")，看不出到底錯在哪。
except MySQLdb.Error as e:
    print("查詢失敗：{}".format(e))

cursor.close()
db.close()

# 使用csv檔案，大量匯入全班成績

In [ ]:
import MySQLdb

# host 必須寫 127.0.0.1，不能寫 localhost。
# MySQLdb 看到 "localhost" 會改走 Unix socket（/tmp/mysql.sock），
# 但 MySQL 跑在 Docker 容器裡，本機沒有那個 socket 檔，會直接連線失敗。
# 寫 127.0.0.1 才會走 TCP，經由 compose.yaml 的 3306 埠對應進容器。
# 四個參數依序是：host, user, password, database（密碼見 mysql-learning/.env）
db = MySQLdb.connect("127.0.0.1", "root", "root", "mydatabase")

cursor = db.cursor()

## 5.將 2-exam_score.csv 所有學生成績插入資料庫
# 原本寫 open('2. exam_score.csv')，但實際檔名是 2-exam_score.csv
#（沒有空格、用連字號），所以會 FileNotFoundError。
with open('2-exam_score.csv', encoding='utf-8') as f:
    for index, i in enumerate(f.readlines()):

        if index == 0:    # 標題列不需導入
            continue

        # CSV 有 8 欄：student_no, name, gender_code, chinese, english,
        # math, social_science, science
        # 但 STUDENTS 資料表沒有 student_no 欄位，INSERT 只有 7 個 %s。
        # 8 個值對 7 個佔位符會拋錯（原本被裸 except 吞掉，變成整批靜默失敗），
        # 所以用 [1:] 切掉第一欄的學號。
        fields = i.strip().split(',')
        student_no = fields[0]
        x = tuple(fields[1:])
        print('insert {} data......'.format(student_no))

        sql = """INSERT INTO STUDENTS
             (NAME, GENDER, CHINESE, ENGLISH, MATH, SOCIAL_SCIENCE, SCIENCE)
             VALUES ( %s, %s, %s, %s, %s, %s, %s)"""

        try:
            cursor.execute(sql, x)
            db.commit()
        except MySQLdb.Error as e:
            db.rollback()
            print('  {} 寫入失敗：{}'.format(student_no, e))

cursor.close()
db.close()
print("CSV 匯入結束")

# 將sophia英文成績改成99

In [ ]:
import MySQLdb

# host 必須寫 127.0.0.1，不能寫 localhost。
# MySQLdb 看到 "localhost" 會改走 Unix socket（/tmp/mysql.sock），
# 但 MySQL 跑在 Docker 容器裡，本機沒有那個 socket 檔，會直接連線失敗。
# 寫 127.0.0.1 才會走 TCP，經由 compose.yaml 的 3306 埠對應進容器。
# 四個參數依序是：host, user, password, database（密碼見 mysql-learning/.env）
db = MySQLdb.connect("127.0.0.1", "root", "root", "mydatabase")

cursor = db.cursor()

## 6.將sophia英文成績改成99
# 原本寫法是 Python 字串格式化把值直接拼進 SQL：
#     sql = "UPDATE STUDENTS SET ENGLISH = 99 WHERE NAME = '%s'" % ('sophia')
# 這裡的 %s 是 Python 的格式化符號，不是 MySQLdb 的佔位符，
# 值沒有經過轉義，名字裡若含單引號就會改變 SQL 語意（SQL injection）。
# 正確做法是把 %s 留在 SQL 裡，值透過 execute 的第二個參數交給驅動去轉義。
sql = "UPDATE STUDENTS SET ENGLISH = 99 WHERE NAME = %s"

try:
    cursor.execute(sql, ('sophia',))
    db.commit()
    # rowcount 是上一次 execute 實際影響的列數，用來確認真的改到人。
    print("更新 {} 列".format(cursor.rowcount))
except MySQLdb.Error as e:
    db.rollback()
    print("更新失敗：{}".format(e))

cursor.close()
db.close()

# 計算各科平均

In [ ]:
import MySQLdb

# host 必須寫 127.0.0.1，不能寫 localhost。
# MySQLdb 看到 "localhost" 會改走 Unix socket（/tmp/mysql.sock），
# 但 MySQL 跑在 Docker 容器裡，本機沒有那個 socket 檔，會直接連線失敗。
# 寫 127.0.0.1 才會走 TCP，經由 compose.yaml 的 3306 埠對應進容器。
# 四個參數依序是：host, user, password, database（密碼見 mysql-learning/.env）
db = MySQLdb.connect("127.0.0.1", "root", "root", "mydatabase")

cursor = db.cursor()

sql = "SELECT * FROM STUDENTS"
chinese_avg = 0
english_avg = 0
math_avg = 0
social_science_avg = 0
science_avg = 0
# 原本 num_students 只在 try 內才被賦值，一旦查詢失敗，
# 最後那幾行 print 會拋 NameError（而且是除以未定義的變數）。
# 先給 0，並在計算平均前檢查，避免 ZeroDivisionError。
num_students = 0

try:

    cursor.execute(sql)

    results = cursor.fetchall()
    num_students = len(results)   # 總共幾個學生

    ## 7.計算各個科目平均成績
    for row in results:

        student_id = row[0]
        student_name = row[1]
        student_gender = row[2]

        student_chinese = row[3]
        chinese_avg = chinese_avg + float(row[3])

        student_english = row[4]
        english_avg = english_avg + float(row[4])

        student_math = row[5]
        math_avg = math_avg + float(row[5])

        student_social_science = row[6]
        social_science_avg = social_science_avg + float(row[6])

        student_science = row[7]
        science_avg = science_avg + float(row[7])

        print('name: {}, gender: {}, chinese: {}, english: {}, math: {}, social_science: {}, science: {}'
               .format(student_name, student_gender, student_chinese, student_english, student_math, student_social_science, student_science))

except MySQLdb.Error as e:
    print("查詢失敗：{}".format(e))

if num_students == 0:
    print('STUDENTS 沒有資料，無法計算平均')
else:
    print('chinese_avg: {}'.format(chinese_avg/num_students))
    print('english_avg: {}'.format(english_avg/num_students))
    print('math_avg: {}'.format(math_avg/num_students))
    print('social_science_avg: {}'.format(social_science_avg/num_students))
    print('science_avg: {}'.format(science_avg/num_students))

cursor.close()
db.close()